# CRYCHIC developer tutorial: subject-aware cross-fit diagnostics

This notebook inspects the checksum-pinned Kang 2018 paired-condition smoke run. CRYCHIC is not a deep-learning model. Outer training and held-out scopes enforce leakage boundaries; inner subject folds select a statistical penalty. They are not neural-network train/validation sets.

The default run intentionally lacks a frozen receiver-autonomous nuisance resource. Its held-out diagnostic scores can be inspected, but official incremental status remains `not_estimable` and the complete pipeline is not OOF-certified.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from benchmarks.adapters.common import write_json
from benchmarks.run_kang2018_ligand_gate_v3 import (
    DEFAULT_CONFIG,
    run_benchmark,
)

In [ ]:
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Run this notebook from the CRYCHIC repository tree')

WORKSPACE_ROOT = Path(os.environ.get('CRYCHIC_WORKSPACE', REPO_ROOT.parent)).resolve()
ARTIFACT_PATH = WORKSPACE_ROOT / 'benchmark_work/kang2018_ligand_gate_v3_default.json'
RUN_BENCHMARK = False  # Set True to refit the bounded real-data smoke (~1 minute, ~2.5 GiB).

{'repo_root': str(REPO_ROOT), 'workspace_root': str(WORKSPACE_ROOT), 'artifact': str(ARTIFACT_PATH)}

## Run or load the frozen smoke

The runner verifies the H5AD checksum, raw-count semantics, subject pairing, complete gene axis, CellChat resource, NicheNet prior, and frozen diagnostic configuration before fitting.

In [ ]:
if RUN_BENCHMARK:
    artifact = run_benchmark(
        workspace_root=WORKSPACE_ROOT,
        config_path=DEFAULT_CONFIG,
    )
    ARTIFACT_PATH.parent.mkdir(parents=True, exist_ok=True)
    write_json(ARTIFACT_PATH, artifact)
elif ARTIFACT_PATH.is_file():
    artifact = json.loads(ARTIFACT_PATH.read_text(encoding='utf-8'))
else:
    raise FileNotFoundError(f'{ARTIFACT_PATH} is missing; set RUN_BENCHMARK=True')

assert artifact['schema_version'] == 'crychic-kang2018-ligand-gate-v3-benchmark-v1'
assert artifact['claims']['certifying'] is False
assert artifact['claims']['method_superiority'] is False
{
    'shape': artifact['input']['analysis_shape'],
    'subjects': artifact['input']['subject_manifest']['n_subjects'],
    'all_subjects_paired': artifact['input']['all_subjects_paired'],
    'elapsed_seconds': artifact['runtime']['total_elapsed_seconds'],
}

## Certification and tuning audit

A diagnostic model may be fitted and applied to outer-heldout subjects while remaining non-official. The two states must not be conflated. Candidate loss summaries are subject-equal and deidentified; no raw subject losses are exported.

In [ ]:
audit = artifact['crossfit_audit']
pd.Series({
    'completed_stage_oof_verified': audit['completed_stage_oof_verified'],
    'complete_pipeline_oof_certified': audit['complete_pipeline_oof_certified'],
    'diagnostic_status_counts': audit['incremental_diagnostic_status_counts'],
    'official_status_counts': audit['incremental_official_status_counts'],
    'remaining_stages': audit['remaining_stages'],
}, name='value')

In [ ]:
tuning_rows = []
for decision in audit['incremental_tuning_records']:
    for candidate in decision['candidate_diagnostics']:
        tuning_rows.append({
            'fold_id': decision['fold_id'],
            'receiver': decision['receiver'],
            'diagnostic_status': decision['diagnostic_status'],
            'official_status': decision['official_status'],
            'lambda1_fraction': candidate['lambda1_fraction'],
            'mean_loss': candidate['mean_subject_equal_loss'],
            'paired_delta_to_best': candidate['mean_paired_loss_difference_to_best'],
            'relative_delta_to_best': candidate['relative_mean_loss_difference_to_best'],
            'paired_delta_se': candidate['paired_difference_standard_error'],
            'within_one_se': candidate['within_paired_one_se'],
            'is_best': candidate['is_best'],
            'is_selected': candidate['is_selected'],
            'max_family_coefficient': decision['maximum_family_coefficient'],
            'n_effective_positive_coefficients': decision['n_effectively_positive_family_coefficients'],
        })
tuning = pd.DataFrame(tuning_rows).sort_values(['fold_id', 'receiver', 'lambda1_fraction'], ascending=[True, True, False])
tuning

## Inspect ligand and receptor gates

Ligand contrast support is learned only from each outer training fold and uses the frozen interaction multiplicity family. Receptor eligibility is a separate training-fold hard gate. A final structural zero can therefore be explained by either component or by downstream family selection.

In [ ]:
ligand = pd.DataFrame(artifact['fold_receiver_interaction_supports'])
receptor = pd.DataFrame(artifact['fold_receiver_interaction_receptor_gates'])
gate_table = ligand.merge(
    receptor[['fold_id', 'receiver', 'interaction_id', 'receptor_gate', 'receptor_gate_threshold', 'receptor_eligible']],
    on=['fold_id', 'receiver', 'interaction_id'],
    how='left',
    validate='one_to_one',
)
gate_table[[
    'fold_id', 'receiver', 'diagnostic_id', 'mean_effect',
    'holm_adjusted_p_value', 'ligand_contrast_gate_status',
    'receptor_gate', 'receptor_gate_threshold', 'receptor_eligible',
]].sort_values(['receiver', 'diagnostic_id', 'fold_id'])

In [ ]:
with plt.style.context(REPO_ROOT / 'benchmarks/report/publication.mplstyle'):
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), layout='constrained')
    ligand_plot = (
        ligand.groupby(['diagnostic_id', 'fold_id'], as_index=False)['mean_effect'].first()
        .pivot(index='diagnostic_id', columns='fold_id', values='mean_effect')
    )
    ligand_plot.plot.bar(ax=axes[0], color=['#3B6FB6', '#D9822B'])
    axes[0].set_title('A  Training-fold ligand effects', loc='left')
    axes[0].set_ylabel('Subject-equal paired effect')
    axes[0].set_xlabel('')
    axes[0].tick_params(axis='x', rotation=25)

    selected = tuning.loc[tuning['lambda1_fraction'].eq(0.1)].copy()
    selected['model'] = selected['receiver'] + ' | ' + selected['fold_id'].str[-6:]
    axes[1].barh(selected['model'], selected['relative_delta_to_best'], color='#2A9D8F')
    axes[1].axvline(0, color='black', linewidth=0.7)
    axes[1].set_title('B  Weaker-penalty loss delta', loc='left')
    axes[1].set_xlabel('Relative paired loss delta to best')
    plt.show()

## Inspect held-out score status

Zero-valued rows are retained with explicit reasons. Do not discard them before checking `status_counts` and `reason_counts`, and do not reinterpret a non-certified diagnostic as an official communication probability.

In [ ]:
member = pd.DataFrame(artifact['paired_summaries']['member']['oof_pooled'])
sender = pd.DataFrame(artifact['paired_summaries']['sender']['oof_pooled'])
member[[
    'receiver', 'interaction_id', 'mode', 'mean_paired_effect',
    'n_complete_subjects', 'status_counts', 'reason_counts',
]].sort_values(['receiver', 'interaction_id', 'mode'])

## Optional: inspect the matched cSCC cross-method smoke

The cSCC summary compares CellChat, CellPhoneDB, LIANA, and CRYCHIC on the same 56-edge LR universe. NicheNet remains a separate source-agnostic Track B proxy. Native result density is output sparsity, not accuracy.

In [ ]:
CSCC_ROOT = WORKSPACE_ROOT / 'benchmark_work/cscc_crossmethod_smoke/summary'
if (CSCC_ROOT / 'track_a_coverage.tsv').is_file():
    cscc_coverage = pd.read_csv(CSCC_ROOT / 'track_a_coverage.tsv', sep='\t')
    cscc_concordance = pd.read_csv(CSCC_ROOT / 'track_a_concordance.tsv', sep='\t')
    display(cscc_coverage[['method_label', 'native_observed_fraction', 'n_estimable_effect_rows']])
    display(cscc_concordance.pivot(index='method_left', columns='method_right', values='spearman').round(3))
else:
    print('Run python -m benchmarks.summarize_cscc_crossmethod_smoke first.')

In [ ]:
assert audit['completed_stage_oof_verified'] is True
assert audit['complete_pipeline_oof_certified'] is False
assert set(audit['incremental_official_status_counts']) == {'not_estimable'}
assert artifact['paired_summaries']['within_run_descriptive_only'] is True
print('Guardrails intact: descriptive diagnostic only; no p/q, causal sender, or superiority claim.')